# Product entity matching: data, features and evaluation

**English** | [简体中文](product-matching-walkthrough.zh-CN.ipynb)

Two retailers may write different titles for the same product. This study compares pairs of listings using text and field comparisons. Follow the saved results, then run small examples to see how the inputs and metrics work.

The final model is named **C+** in this project: RoBERTa reads the paired product text, combines it with 42 field comparisons, and averages match probabilities across three training seeds. The comparison table below defines each method.

The measured results concern **555 fixed Walmart–Amazon pairs**. This is a retrospective comparison: historical exposure is incompletely documented, and performance on independent new Walmart–Amazon samples has not been tested. The Acme listings, labels and probabilities below are invented teaching examples, not model predictions.

**Read or run:** GitHub displays the saved outputs, so you can read the walkthrough without setting anything up. To run cells interactively, use an existing Jupyter Python 3.11+ kernel and run them in order. As an alternative that needs no package installation, follow the [README quickstart](../README.md#quickstart) to create the lightweight project environment and run `scripts/run_notebook.py` with its explicit interpreter. It executes both translations with the standard library and checks their saved outputs. These cells need no product dataset or model download.

[Project overview](../README.md) · [Data and results](../showcase/en/index.html) · [Reproduction guide](../docs/REPRODUCIBILITY.md)


In [1]:
from pathlib import Path
import json, math, sys
LANGUAGE = "en"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "results/fixed-results.json").is_file() and (p / "src/product_matching").is_dir())
sys.path.insert(0, str(ROOT / "src"))
facts = json.loads((ROOT / "results/fixed-results.json").read_text(encoding="utf-8"))
text = json.loads((ROOT / "notebooks/walkthrough_text.json").read_text(encoding="utf-8"))[LANGUAGE]
print(text["task"])
for role, count in facts["roles"].items():
    print(f'{text["roles"][role]}: {count:,}')


Task: decide whether two retailer listings refer to the same product.
Fit pairs: 3,738
Development pairs: 199
Calibration pairs: 209
Evaluation pairs: 555


## 1. Data roles and label separation

| Role | Pairs | Use |
| --- | ---: | --- |
| Fit | 3,738 | Update model parameters and fit title IDF |
| Development | 199 | Select a shared checkpoint epoch |
| Calibration | 209 | Select a classification cutoff |
| Evaluation | 555 | Measure the frozen models and cutoffs |

The evaluation pool contains **190 matches and 365 nonmatches**. IDF (inverse document frequency) gives more weight to words that occur in fewer fit titles, reducing the influence of common words on title similarity.

A `pair_id` aligns a prediction with its label; row order is not treated as identity. Only **fit model inputs** carry target labels. Non-fit labels are kept separate from prediction inputs and joined when selecting checkpoints, calibrating the cutoff or computing metrics.

The pipeline is `fit-only IDF → role preparation → development selection → calibration cutoff → evaluation`. These roles describe intended use; they do not establish that the historical evaluation pool was untouched.


## 2. Fixed-pool model comparison

All six methods used the same 555-pair evaluation pool. Their names refer to the following models and comparison features:

| Method | Description |
| --- | --- |
| B22 | Logistic regression on 22 comparison features |
| L30 | Histogram gradient-boosted trees on 30 comparison features |
| S33 | L30 plus three fixed MiniLM semantic/missingness features |
| L42 | Histogram gradient-boosted trees on all 42 comparison features |
| C+ | RoBERTa with 42 comparison features, averaged over three training seeds |
| C0 | Independently trained neural control with the 42-feature input zeroed throughout training and inference |

The next cell loads **saved aggregate results**. Read **AP** (average precision) as ranking quality, **F1** as the balance of false and missed matches at each method's calibration cutoff, and **P@100** as precision among the first 100 ranked pairs. AP is not classification accuracy.


In [2]:
print(f'{text["method"]:<8} {"AP":>9} {"F1":>9} {"P@100":>9}')
for method, result in facts["metrics"].items():
    print(f'{method:<8} {result["ap"]:9.6f} {result["f1"]:9.6f} {result["p_at_100"]:9.2f}')


Method          AP        F1     P@100
B22       0.795590  0.743719      0.81
L30       0.844899  0.762115      0.90
S33       0.846702  0.767123      0.89
L42       0.949700  0.852041      1.00
C+        0.965998  0.901639      1.00
C0        0.965616  0.875648      1.00


### Absolute differences and error counts

L42 is the strongest tabular reference in this pool. Subtract its score from C+, then multiply by 100 to express an absolute difference in percentage points. Compare the independently trained C0 as well. The saved false-match and missed-match counts help explain what changes at each method's calibration cutoff.


In [3]:
cplus, reference, ablation = (facts["metrics"][m] for m in ("C+", "L42", "C0"))
print(f'{text["ap_gain"]}: {(cplus["ap"]-reference["ap"])*100:.4f} {text["pp"]}')
print(f'{text["f1_gain"]}: {(cplus["f1"]-reference["f1"])*100:.4f} {text["pp"]}')
print(f'{text["ap_c0"]}: {(cplus["ap"]-ablation["ap"])*100:.4f} {text["pp"]}')
print(f'{text["f1_c0"]}: {(cplus["f1"]-ablation["f1"])*100:.4f} {text["pp"]}')
for method, result in (("L42", reference), ("C+", cplus)):
    counts = result["confusion"]
    print(f'{method} {text["error_counts"]}: FP={counts["fp"]}, FN={counts["fn"]}')


AP gain over L42: 1.6298 percentage points
F1 gain over L42: 4.9599 percentage points
AP difference versus C0: 0.0381 percentage points
F1 difference versus C0 at separately calibrated cutoffs: 2.5992 percentage points
L42 saved error counts: FP=35, FN=23
C+ saved error counts: FP=11, FN=25


At their respective calibration cutoffs, C+ reduces false matches from **35 to 11** compared with L42, while missed matches rise from **23 to 25**. Precision improves and recall falls slightly; the higher F1 does not mean that both kinds of error decrease. L42, C+ and C0 all have P@100 = 1.00, so that metric does not distinguish their first 100 ranked pairs on this pool.

C+ and C0 differ by about **0.04 percentage points in AP**, while C+'s F1 is about **2.60 percentage points higher at their separately calibrated cutoffs**. These independently trained models do not establish a large feature-driven AP contribution or a statistically significant effect. The comparison describes this fixed pool; independent fresh-sample performance remains untested.


## 3. Feature construction for synthetic product pairs

The invented Acme pair differs in model-number punctuation, title spacing and price. Two variants change the right-hand model from `ZX100` to `ZX101`, or change its currency from USD to EUR. The table below compares all three using the **same feature function and the toy-fit IDF state created here**.

The 42 features combine **14 business comparisons + 8 title comparisons (4 fit-only IDF-weighted comparisons and 4 direct comparisons) + 8 title-code comparisons + 12 native model/category comparisons**. Business fields include title, brand, description, price and currency. Native fields are the source-provided `modelno` and `category`, kept separately from the business view. See the [data card and feature dictionary](../docs/DATA_CARD.md) for all 42 identifiers.

The displayed features are binary flags: 1 means the named condition holds, and 0 means it does not. Compact-title and compact-model equality ignore spacing and punctuation. The different-model row also flags **same model letters but different digits**; the EUR row makes the USD/EUR price comparison unavailable. A missing price stays missing rather than becoming an observed zero price.

The toy fit label only exercises the input schema and IDF construction. **No classifier is trained, and RoBERTa does not score these records.** These field comparisons are evidence, not a final match decision. See the [code map](../docs/CODE_MAP.md) for the implementation.


In [4]:
from product_matching.data import business_record, native_record
from product_matching.features import FEATURE_NAMES_42, fit_title_evidence, prepare_rows
raw_left = {"title":"Acme ZX-100 16GB", "brand":"Acme", "price":"19.99", "priceCurrency":"USD"}
raw_right = {"title":"ACME ZX100 16 GB", "brand":"ACME", "price":"20.00", "priceCurrency":"USD"}
example = {"pair_id":"synthetic:acme", "left":business_record(raw_left), "right":business_record(raw_right),
           "left_native":native_record({"modelno":"ZX-100", "category":"electronics"}),
           "right_native":native_record({"modelno":"ZX100", "category":"electronics"})}
fit_idf = fit_title_evidence([{**example, "label":1}])
different_model = {
    **example, "pair_id":"synthetic:acme-different-model",
    "right":business_record({**raw_right, "title":"ACME ZX101 16 GB"}),
    "right_native":native_record({"modelno":"ZX101", "category":"electronics"}),
}
different_currency = {
    **example, "pair_id":"synthetic:acme-different-currency",
    "right":business_record({**raw_right, "priceCurrency":"EUR"}),
}
prepared = prepare_rows([example, different_model, different_currency], role="dev", fit_idf=fit_idf)
feature_values = [dict(zip(FEATURE_NAMES_42, row["x42"])) for row in prepared]
columns = ("price_comparable", "title_compact_exact", "native_model_compact_exact",
           "native_model_same_letters_different_digits")
print(text["feature_width"], ":", len(feature_values[0]))
print(" | ".join([text["case"], *[text["features"][key] for key in columns]]))
for case, row, values in zip(("same_model", "different_model", "different_currency"), prepared, feature_values):
    print(" | ".join([text["cases"][case], *[f"{values[key]:.0f}" for key in columns]]))
    assert len(values)==42 and "label" not in row
assert feature_values[0]["native_model_compact_exact"] == 1
assert feature_values[1]["native_model_compact_exact"] == 0
assert feature_values[1]["native_model_same_letters_different_digits"] == 1
assert feature_values[2]["price_comparable"] == 0


Synthetic feature count : 42
Synthetic input | Comparable prices | Equal compact titles | Equal compact model numbers | Same model letters, different digits
Same model number | 1 | 1 | 1 | 0
Different model digits | 1 | 0 | 0 | 1
Different currency | 0 | 1 | 1 | 0


### Input-role validation

Try adding a target label to an evaluation input. The interface should reject it and leave the fit-only IDF state unchanged. Passing this check demonstrates the current interface rule; it cannot retroactively prove that all historical data use was leakage-free.


In [5]:
from product_matching.errors import AdmissionError
before = fit_idf.payload()
try:
    prepare_rows([{**example, "label":1}], role="evaluation", fit_idf=fit_idf)
except AdmissionError:
    print(text["label_rejected"])
else:
    raise AssertionError(text["label_accepted"])
assert before == fit_idf.payload()
print(text["idf_unchanged"])


The evaluation input rejected a target label, as expected.
The fit-only IDF state is unchanged.


## 4. AB/BA directions and three-seed aggregation

A and B are the two listings. RoBERTa-base reads the pair in both AB and BA order. A 64-unit projection is concatenated with 42 comparison features; another 64-unit GELU layer, dropout and a scalar output produce a logit, the raw score before sigmoid converts it to a probability. For each seed, the AB/BA logits are averaged before sigmoid. The three seed probabilities are then averaged equally.

C+ follows a six-epoch training schedule; development data selects the shared epoch-4 checkpoints for the ensemble. All three seeds remain in the ensemble. Training used BF16 encoder autocast with FP32 parameters and head; inference used FP32. See the [model card](../docs/MODEL_CARD.md) for the fixed configuration.

The next cell uses **invented probabilities 0.9, 0.3 and 0.3** to explain the arithmetic. They are not RoBERTa outputs or measured seed results.


In [6]:
from product_matching.metrics import mean_scores
invented = {seed:[{"pair_id":"synthetic:acme", "score":value}]
            for seed,value in zip((42,43,44),(.9,.3,.3))}
mean = mean_scores(["synthetic:acme"], invented)[0]["score"]
print(text["mean"], ":", mean)
print(text["seeds"], ":", facts["model"]["seeds"], ";", text["epoch"], ":", facts["model"]["epoch"])
assert mean == .5


Invented probabilities: equal mean : 0.5
Study seeds : [42, 43, 44] ; Shared epoch : 4


## 5. Ranking metrics, cutoff and confusion counts

AP combines precision and recall changes along the ranking. Equal scores are handled together, so it does not award an arbitrary advantage to whichever tied item happens to appear first. In the invented example below, the top tie contains one match and one non-match. Its AP is 0.583333; that is not a classification accuracy.

The real experiment selected its cutoff on calibration data by F1, breaking ties by precision and then the higher cutoff. Evaluation did not choose a new cutoff. After the synthetic AP example, the cell reads C+'s **saved aggregate confusion counts** and independently recovers F1 as `2 × TP / (2 × TP + FP + FN)`.

Here, TP means found matches, FP means false matches, FN means missed matches, and TN means correctly rejected non-matches. Changing the cutoff can change these counts and F1; it does not change the underlying ranking AP.


In [7]:
from product_matching.metrics import average_precision
toy = [{"pair_id":"synthetic:a", "label":1, "score":.9},
       {"pair_id":"synthetic:b", "label":0, "score":.9},
       {"pair_id":"synthetic:c", "label":1, "score":.5}]
print(f'{text["toy_ap"]}: {average_precision(toy):.6f}')
print(text["threshold"], ":", cplus["threshold"])
confusion = cplus["confusion"]
tp,fp,fn,tn = (confusion[k] for k in ("tp","fp","fn","tn"))
f1_from_counts = 2*tp/(2*tp+fp+fn)
assert math.isclose(f1_from_counts, cplus["f1"], rel_tol=0, abs_tol=1e-14)
print(f'{text["confusion"]}: TP={tp}, FP={fp}, FN={fn}, TN={tn}')
print(f'{text["f1_recovered"]}: {f1_from_counts:.6f}')


Invented scores: AP with a tie: 0.583333
Saved C+ calibration cutoff : 0.3947772259513537
Saved aggregate counts: TP=165, FP=11, FN=25, TN=354
F1 recovered from the saved counts: 0.901639


## 6. Reproduction scope and execution commands

| Activity | What the evidence covers |
| --- | --- |
| Public checks and this Notebook | Saved aggregate arithmetic, input rules and synthetic examples; no product dataset or weights needed |
| Existing-weight replay | Saved model predictions reproduced without differences on a Windows RTX 3060 laptop, using locally supplied original inputs and artifacts |
| Raw-source role reconstruction | Four business input files and their member order matched the saved originals; this restores the original membership |
| Full experiment from scratch | Fresh end-to-end training has not yet been validated |

These cells cover the first row. Existing-weight replay loads already trained models; it does not repeat training or evaluate an independent new sample. Role reconstruction restores the historical inputs and does not resolve their incomplete exposure record. The public repository does not distribute product records or trained weights.

Create the lightweight project environment as shown in the [README quickstart](../README.md#quickstart), then run these checks from the repository root using its explicit interpreter:

| Platform | Public checks | Notebook |
| --- | --- | --- |
| macOS / Linux | `.venv/bin/python scripts/public_smoke.py` | `.venv/bin/python scripts/run_notebook.py` |
| Windows | `.\.venv\Scripts\python.exe scripts/public_smoke.py` | `.\.venv\Scripts\python.exe scripts/run_notebook.py` |

Existing-weight prediction requires locally supplied product inputs, model snapshots and trained weights. Separate prediction and fresh-training commands, along with the detailed replay evidence, are in the [reproduction guide](../docs/REPRODUCIBILITY.md).


In [8]:
for key in ("max_prediction_difference", "new_training_in_replay", "fresh_training_reproduced"):
    value = facts["reproduction"][key]
    display = text["yes"] if value is True else text["no"] if value is False else value
    print(f'{text["evidence"][key]}: {display}')
heavy = {"torch", "transformers", "sklearn", "sentence_transformers"}
assert not heavy & set(sys.modules)
print(text["done"])


Saved replay: maximum prediction difference: 0.0
New training during the replay: no
Full retraining validated: no
Walkthrough complete: aggregate arithmetic and synthetic feature/metric examples.


## Further reading

Read the [data card](../docs/DATA_CARD.md) for the fields and all 42 comparisons, the [model card](../docs/MODEL_CARD.md) for the full configuration, and the [limitations](../docs/LIMITATIONS.md) for what the fixed-pool evidence can support. The [project card](../docs/PROJECT_CARD.md) gives a compact overview.
